In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

current_dir = Path.cwd()

if current_dir.name.lower() == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

DATA_DIR = PROJECT_ROOT / "data"
AUDIT_DIR = DATA_DIR / "audit"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_FILE_INVENTORY_CSV = AUDIT_DIR / "project_file_inventory_v0_1.csv"
PROJECT_CLEANUP_CANDIDATES_CSV = AUDIT_DIR / "project_cleanup_candidates_v0_1.csv"

print("Project root:", PROJECT_ROOT)
print("Data dir exists:", DATA_DIR.exists())
print("Audit dir:", AUDIT_DIR)

Project root: C:\Users\patri\vrp_project
Data dir exists: True
Audit dir: C:\Users\patri\vrp_project\data\audit


In [2]:
# ============================================================
# Build project file inventory
# ============================================================

file_rows = []

for path in PROJECT_ROOT.rglob("*"):
    if path.is_file():
        stat = path.stat()

        file_rows.append({
            "file_path": str(path),
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "parent_folder": str(path.parent.relative_to(PROJECT_ROOT)),
            "file_name": path.name,
            "suffix": path.suffix.lower(),
            "size_mb": stat.st_size / (1024 * 1024),
            "modified_time": pd.Timestamp(stat.st_mtime, unit="s"),
        })

inventory_df = pd.DataFrame(file_rows)

inventory_df["top_level_folder"] = inventory_df["relative_path"].str.split("\\").str[0]
inventory_df["size_gb"] = inventory_df["size_mb"] / 1024

print("Total files:", len(inventory_df))
print("Total size GB:", inventory_df["size_gb"].sum())

print("\nSize by top-level folder:")
display(
    inventory_df
    .groupby("top_level_folder")["size_mb"]
    .sum()
    .sort_values(ascending=False)
    .rename("size_mb")
    .to_frame()
)

print("\nSize by suffix:")
display(
    inventory_df
    .groupby("suffix")["size_mb"]
    .agg(["count", "sum"])
    .sort_values("sum", ascending=False)
)

print("\nLargest 50 files:")
display(
    inventory_df
    .sort_values("size_mb", ascending=False)
    .head(50)
)

Total files: 11378
Total size GB: 0.8140868796035647

Size by top-level folder:


,size_mb
top_level_folder,
data,798.338392
notebooks,33.867492
Trades.xlsx,1.419081



Size by suffix:


,count,sum
suffix,,
.pkl,11233,730.669843
.csv,71,71.599348
.parquet,14,16.667240
.ipynb,59,13.269453
.xlsx,1,1.419081



Largest 50 files:


,file_path,relative_path,parent_folder,file_name,suffix,size_mb,modified_time,top_level_folder,size_gb
463,C:\Users\patri\vrp_project\data\processed\trad...,data\processed\trade_signal_panel_v0_1.csv,data\processed,trade_signal_panel_v0_1.csv,.csv,15.642781,2026-06-28 16:53:02.796918869,data,0.015276
471,C:\Users\patri\vrp_project\data\processed\vrp_...,data\processed\vrp_zscore_panel_v0_1.csv,data\processed,vrp_zscore_panel_v0_1.csv,.csv,14.564443,2026-06-28 12:48:56.734323740,data,0.014223
469,C:\Users\patri\vrp_project\data\processed\vrp_...,data\processed\vrp_panel_v0_1.csv,data\processed,vrp_panel_v0_1.csv,.csv,12.067234,2026-06-28 02:10:45.194436312,data,0.011784
467,C:\Users\patri\vrp_project\data\processed\vix_...,data\processed\vix_term_structure_history_v0_7...,data\processed,vix_term_structure_history_v0_7_1_repaired_tot...,.csv,6.958735,2026-06-28 00:49:47.931168556,data,0.006796
11376,C:\Users\patri\vrp_project\data\processed\arch...,data\processed\archive\vix_term_structure_hist...,data\processed\archive,vix_term_structure_history_v0_7_1_repaired.csv,.csv,6.196209,2026-06-27 22:14:02.835753202,data,0.006051
465,C:\Users\patri\vrp_project\data\processed\vix_...,data\processed\vix_term_structure_history.csv,data\processed,vix_term_structure_history.csv,.csv,5.400764,2026-06-27 21:15:24.108476400,data,0.005274
464,C:\Users\patri\vrp_project\data\processed\trad...,data\processed\trade_signal_panel_v0_1.parquet,data\processed,trade_signal_panel_v0_1.parquet,.parquet,4.275062,2026-06-28 16:53:02.915691853,data,0.004175
472,C:\Users\patri\vrp_project\data\processed\vrp_...,data\processed\vrp_zscore_panel_v0_1.parquet,data\processed,vrp_zscore_panel_v0_1.parquet,.parquet,4.253613,2026-06-28 12:48:56.879493713,data,0.004154
470,C:\Users\patri\vrp_project\data\processed\vrp_...,data\processed\vrp_panel_v0_1.parquet,data\processed,vrp_panel_v0_1.parquet,.parquet,3.077864,2026-06-28 02:10:45.309486151,data,0.003006
452,C:\Users\patri\vrp_project\data\processed\real...,data\processed\realized_variance_panel_v0_1.csv,data\processed,realized_variance_panel_v0_1.csv,.csv,1.835705,2026-06-28 02:10:44.289603233,data,0.001793


In [3]:
# ============================================================
# Classify files for cleanup review
# ============================================================

CANONICAL_KEEP_FILES = {
    # VIX / term structure
    "data\\processed\\vix_term_structure_history_v0_7_1_repaired_total_variance.parquet",

    # Realized / VRP
    "data\\processed\\realized_variance_panel_v0_1.parquet",
    "data\\processed\\vrp_panel_v0_1.parquet",

    # Z-scores / preferred tenor
    "data\\processed\\vrp_zscore_panel_v0_1.parquet",
    "data\\processed\\richest_vrp_tenor_by_date_v0_1.parquet",
    "data\\processed\\preferred_vrp_tenor_by_date_v0_1.parquet",

    # Signals
    "data\\processed\\trade_signal_panel_v0_1.parquet",
    "data\\processed\\daily_trade_signal_summary_v0_1.parquet",

    # Candidates / pricing
    "data\\processed\\trade_candidate_panel_v0_1.parquet",
    "data\\processed\\trade_candidate_pricing_v0_1.parquet",
}

def normalize_rel_path(path_text):
    return str(path_text).replace("/", "\\")

def classify_file(row):
    rel = normalize_rel_path(row["relative_path"])
    suffix = row["suffix"]
    name = row["file_name"]
    folder = normalize_rel_path(row["parent_folder"])

    # Never touch environment, git, or hidden/system-ish files in this audit.
    if (
        ".git" in rel
        or ".venv" in rel
        or "__pycache__" in rel
        or ".ipynb_checkpoints" in rel
    ):
        return "DO_NOT_TOUCH", "environment/cache/system/project metadata"

    # Raw ThetaData chain cache: valuable for repricing and later strategy variants.
    if rel.startswith("data\\raw\\thetadata_chains\\"):
        return "KEEP_FOR_NOW", "raw ThetaData chain cache needed for repricing and alternative tests"

    # Canonical parquet files.
    if rel in CANONICAL_KEEP_FILES:
        return "KEEP", "canonical project output"

    # Audit CSVs are small and useful.
    if rel.startswith("data\\audit\\"):
        return "KEEP", "audit file, usually small and useful"

    # Processed CSVs that duplicate parquet outputs are cleanup candidates.
    if rel.startswith("data\\processed\\") and suffix == ".csv":
        parquet_equivalent = Path(str(row["file_path"])).with_suffix(".parquet")
        if parquet_equivalent.exists():
            return "DELETE_OPTIONAL", "CSV duplicate of parquet file"
        else:
            return "ARCHIVE_OPTIONAL", "processed CSV without matching parquet; review before deleting"

    # Old unrepaired vix file.
    if rel in {
        "data\\processed\\vix_term_structure_history.parquet",
        "data\\processed\\vix_term_structure_history.csv",
    }:
        return "ARCHIVE_OPTIONAL", "old pre-repair v0.7 term-structure file"

    # Raw target checkpoints from notebook 08.
    if "trade_candidate_panel_v0_1_raw_targets" in name:
        return "DELETE_OPTIONAL", "intermediate raw target checkpoint; final mapped candidate file exists"

    # Notebook checkpoint files.
    if ".ipynb_checkpoints" in rel:
        return "DELETE_OPTIONAL", "Jupyter checkpoint"

    # Large temporary or backup files.
    if suffix in [".tmp", ".bak", ".old"]:
        return "DELETE_OPTIONAL", "temporary/backup file"

    # Notebook files should usually be kept unless obvious duplicates.
    if suffix == ".ipynb":
        if "clean" in name.lower():
            return "KEEP", "cleaned notebook"
        else:
            return "REVIEW", "notebook; review before deleting"

    # Parquet outputs not explicitly canonical should be reviewed, not deleted.
    if rel.startswith("data\\processed\\") and suffix == ".parquet":
        return "REVIEW", "processed parquet not in canonical keep list"

    return "REVIEW", "unclassified file; review manually"

classification_results = inventory_df.apply(classify_file, axis=1, result_type="expand")
inventory_df["cleanup_classification"] = classification_results[0]
inventory_df["cleanup_reason"] = classification_results[1]

print("Classification counts:")
display(
    inventory_df
    .groupby("cleanup_classification")
    .agg(
        files=("file_path", "count"),
        size_mb=("size_mb", "sum"),
    )
    .sort_values("size_mb", ascending=False)
)

print("\nLargest cleanup/review candidates:")
display(
    inventory_df[
        inventory_df["cleanup_classification"].isin(
            ["DELETE_OPTIONAL", "ARCHIVE_OPTIONAL", "REVIEW"]
        )
    ]
    .sort_values("size_mb", ascending=False)
    .head(100)
)

Classification counts:


,files,size_mb
cleanup_classification,,
KEEP_FOR_NOW,10901,710.075993
DELETE_OPTIONAL,15,68.323206
REVIEW,352,27.506744
KEEP,79,20.612312
DO_NOT_TOUCH,28,6.438798
ARCHIVE_OPTIONAL,3,0.667912



Largest cleanup/review candidates:


,file_path,relative_path,parent_folder,file_name,suffix,size_mb,modified_time,top_level_folder,size_gb,cleanup_classification,cleanup_reason
463,C:\Users\patri\vrp_project\data\processed\trad...,data\processed\trade_signal_panel_v0_1.csv,data\processed,trade_signal_panel_v0_1.csv,.csv,15.642781,2026-06-28 16:53:02.796918869,data,0.015276,DELETE_OPTIONAL,CSV duplicate of parquet file
471,C:\Users\patri\vrp_project\data\processed\vrp_...,data\processed\vrp_zscore_panel_v0_1.csv,data\processed,vrp_zscore_panel_v0_1.csv,.csv,14.564443,2026-06-28 12:48:56.734323740,data,0.014223,DELETE_OPTIONAL,CSV duplicate of parquet file
469,C:\Users\patri\vrp_project\data\processed\vrp_...,data\processed\vrp_panel_v0_1.csv,data\processed,vrp_panel_v0_1.csv,.csv,12.067234,2026-06-28 02:10:45.194436312,data,0.011784,DELETE_OPTIONAL,CSV duplicate of parquet file
467,C:\Users\patri\vrp_project\data\processed\vix_...,data\processed\vix_term_structure_history_v0_7...,data\processed,vix_term_structure_history_v0_7_1_repaired_tot...,.csv,6.958735,2026-06-28 00:49:47.931168556,data,0.006796,DELETE_OPTIONAL,CSV duplicate of parquet file
11376,C:\Users\patri\vrp_project\data\processed\arch...,data\processed\archive\vix_term_structure_hist...,data\processed\archive,vix_term_structure_history_v0_7_1_repaired.csv,.csv,6.196209,2026-06-27 22:14:02.835753202,data,0.006051,DELETE_OPTIONAL,CSV duplicate of parquet file
...,...,...,...,...,...,...,...,...,...,...,...
381,C:\Users\patri\vrp_project\notebooks\data\raw\...,notebooks\data\raw\thetadata_chains\SPX_202403...,notebooks\data\raw\thetadata_chains,SPX_20240314_20240419_160000.pkl,.pkl,0.093412,2026-06-26 17:54:40.817772865,notebooks,0.000091,REVIEW,unclassified file; review manually
382,C:\Users\patri\vrp_project\notebooks\data\raw\...,notebooks\data\raw\thetadata_chains\SPX_202403...,notebooks\data\raw\thetadata_chains,SPX_20240315_20240419_160000.pkl,.pkl,0.093412,2026-06-26 17:10:35.053785801,notebooks,0.000091,REVIEW,unclassified file; review manually
384,C:\Users\patri\vrp_project\notebooks\data\raw\...,notebooks\data\raw\thetadata_chains\SPX_202403...,notebooks\data\raw\thetadata_chains,SPX_20240319_20240419_160000.pkl,.pkl,0.093412,2026-06-26 17:55:06.293417215,notebooks,0.000091,REVIEW,unclassified file; review manually
335,C:\Users\patri\vrp_project\notebooks\data\raw\...,notebooks\data\raw\thetadata_chains\SPX_202401...,notebooks\data\raw\thetadata_chains,SPX_20240122_20240216_160000.pkl,.pkl,0.093177,2026-06-26 17:26:40.953311682,notebooks,0.000091,REVIEW,unclassified file; review manually


In [4]:
# ============================================================
# Create cleanup candidate report
# ============================================================

cleanup_candidates_df = (
    inventory_df[
        inventory_df["cleanup_classification"].isin(
            ["DELETE_OPTIONAL", "ARCHIVE_OPTIONAL", "REVIEW"]
        )
    ]
    .copy()
    .sort_values(["cleanup_classification", "size_mb"], ascending=[True, False])
)

columns_to_show = [
    "cleanup_classification",
    "cleanup_reason",
    "relative_path",
    "suffix",
    "size_mb",
    "modified_time",
]

display(cleanup_candidates_df[columns_to_show].head(200))

inventory_df.to_csv(PROJECT_FILE_INVENTORY_CSV, index=False)
cleanup_candidates_df.to_csv(PROJECT_CLEANUP_CANDIDATES_CSV, index=False)

print("Saved inventory:", PROJECT_FILE_INVENTORY_CSV)
print("Saved cleanup candidates:", PROJECT_CLEANUP_CANDIDATES_CSV)

,cleanup_classification,cleanup_reason,relative_path,suffix,size_mb,modified_time
466,ARCHIVE_OPTIONAL,old pre-repair v0.7 term-structure file,data\processed\vix_term_structure_history.parquet,.parquet,0.613237,2026-06-27 21:15:24.159845591
449,ARCHIVE_OPTIONAL,processed CSV without matching parquet; review...,data\processed\fred_sofr_history.csv,.csv,0.034235,2026-06-26 19:11:08.916168213
456,ARCHIVE_OPTIONAL,processed CSV without matching parquet; review...,data\processed\spx_trading_dates.csv,.csv,0.020439,2026-06-27 02:12:18.764760494
463,DELETE_OPTIONAL,CSV duplicate of parquet file,data\processed\trade_signal_panel_v0_1.csv,.csv,15.642781,2026-06-28 16:53:02.796918869
471,DELETE_OPTIONAL,CSV duplicate of parquet file,data\processed\vrp_zscore_panel_v0_1.csv,.csv,14.564443,2026-06-28 12:48:56.734323740
...,...,...,...,...,...,...
134,REVIEW,unclassified file; review manually,notebooks\data\raw\thetadata_chains\SPXW_20240...,.pkl,0.048794,2026-06-26 17:27:51.086629868
139,REVIEW,unclassified file; review manually,notebooks\data\raw\thetadata_chains\SPXW_20240...,.pkl,0.048794,2026-06-26 17:28:07.403972387
122,REVIEW,unclassified file; review manually,notebooks\data\raw\thetadata_chains\SPXW_20240...,.pkl,0.048556,2026-06-26 17:27:09.197649717
211,REVIEW,unclassified file; review manually,notebooks\data\raw\thetadata_chains\SPXW_20240...,.pkl,0.048556,2026-06-26 17:45:54.576832533


Saved inventory: C:\Users\patri\vrp_project\data\audit\project_file_inventory_v0_1.csv
Saved cleanup candidates: C:\Users\patri\vrp_project\data\audit\project_cleanup_candidates_v0_1.csv


In [5]:
# ============================================================
# Summarize likely safe deletions
# ============================================================

delete_optional_df = inventory_df[
    inventory_df["cleanup_classification"] == "DELETE_OPTIONAL"
].copy()

archive_optional_df = inventory_df[
    inventory_df["cleanup_classification"] == "ARCHIVE_OPTIONAL"
].copy()

review_df = inventory_df[
    inventory_df["cleanup_classification"] == "REVIEW"
].copy()

print("DELETE_OPTIONAL files:", len(delete_optional_df))
print("DELETE_OPTIONAL size MB:", delete_optional_df["size_mb"].sum())

print("\nARCHIVE_OPTIONAL files:", len(archive_optional_df))
print("ARCHIVE_OPTIONAL size MB:", archive_optional_df["size_mb"].sum())

print("\nREVIEW files:", len(review_df))
print("REVIEW size MB:", review_df["size_mb"].sum())

print("\nDELETE_OPTIONAL detail:")
display(
    delete_optional_df[
        [
            "relative_path",
            "size_mb",
            "cleanup_reason",
        ]
    ]
    .sort_values("size_mb", ascending=False)
)

print("\nARCHIVE_OPTIONAL detail:")
display(
    archive_optional_df[
        [
            "relative_path",
            "size_mb",
            "cleanup_reason",
        ]
    ]
    .sort_values("size_mb", ascending=False)
)

DELETE_OPTIONAL files: 15
DELETE_OPTIONAL size MB: 68.32320594787598

ARCHIVE_OPTIONAL files: 3
ARCHIVE_OPTIONAL size MB: 0.6679115295410156

REVIEW files: 352
REVIEW size MB: 27.506744384765625

DELETE_OPTIONAL detail:


,relative_path,size_mb,cleanup_reason
463,data\processed\trade_signal_panel_v0_1.csv,15.642781,CSV duplicate of parquet file
471,data\processed\vrp_zscore_panel_v0_1.csv,14.564443,CSV duplicate of parquet file
469,data\processed\vrp_panel_v0_1.csv,12.067234,CSV duplicate of parquet file
467,data\processed\vix_term_structure_history_v0_7...,6.958735,CSV duplicate of parquet file
11376,data\processed\archive\vix_term_structure_hist...,6.196209,CSV duplicate of parquet file
465,data\processed\vix_term_structure_history.csv,5.400764,CSV duplicate of parquet file
452,data\processed\realized_variance_panel_v0_1.csv,1.835705,CSV duplicate of parquet file
461,data\processed\trade_candidate_pricing_v0_1.csv,1.422599,CSV duplicate of parquet file
457,data\processed\trade_candidate_panel_v0_1.csv,1.031248,CSV duplicate of parquet file
454,data\processed\richest_vrp_tenor_by_date_v0_1.csv,1.012476,CSV duplicate of parquet file



ARCHIVE_OPTIONAL detail:


,relative_path,size_mb,cleanup_reason
466,data\processed\vix_term_structure_history.parquet,0.613237,old pre-repair v0.7 term-structure file
449,data\processed\fred_sofr_history.csv,0.034235,processed CSV without matching parquet; review...
456,data\processed\spx_trading_dates.csv,0.020439,processed CSV without matching parquet; review...


In [6]:
# ============================================================
# Controlled cleanup: delete DELETE_OPTIONAL files only
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

CLEANUP_CANDIDATES_CSV = AUDIT_DIR / "project_cleanup_candidates_v0_1.csv"
DELETE_LOG_CSV = AUDIT_DIR / "project_cleanup_delete_log_v0_1.csv"

cleanup_candidates_df = pd.read_csv(CLEANUP_CANDIDATES_CSV)

delete_df = cleanup_candidates_df[
    cleanup_candidates_df["cleanup_classification"] == "DELETE_OPTIONAL"
].copy()

print("Files marked DELETE_OPTIONAL:", len(delete_df))
print("Total size MB:", delete_df["size_mb"].sum())

display(
    delete_df[
        [
            "relative_path",
            "size_mb",
            "cleanup_reason",
        ]
    ].sort_values("size_mb", ascending=False)
)

# Safety checks
if delete_df.empty:
    raise ValueError("No DELETE_OPTIONAL files found.")

for rel_path in delete_df["relative_path"]:
    rel_path_str = str(rel_path).replace("/", "\\")
    
    if ".venv" in rel_path_str or ".git" in rel_path_str or "__pycache__" in rel_path_str:
        raise ValueError(f"Safety stop: refusing to delete environment/system file: {rel_path_str}")
    
    if not (
        rel_path_str.startswith("data\\processed\\")
        or rel_path_str.startswith("data\\processed\\archive\\")
    ):
        raise ValueError(f"Safety stop: file outside allowed processed-data folders: {rel_path_str}")

print("Safety checks passed.")

Files marked DELETE_OPTIONAL: 15
Total size MB: 68.32320594787598


,relative_path,size_mb,cleanup_reason
3,data\processed\trade_signal_panel_v0_1.csv,15.642781,CSV duplicate of parquet file
4,data\processed\vrp_zscore_panel_v0_1.csv,14.564443,CSV duplicate of parquet file
5,data\processed\vrp_panel_v0_1.csv,12.067234,CSV duplicate of parquet file
6,data\processed\vix_term_structure_history_v0_7...,6.958735,CSV duplicate of parquet file
7,data\processed\archive\vix_term_structure_hist...,6.196209,CSV duplicate of parquet file
8,data\processed\vix_term_structure_history.csv,5.400764,CSV duplicate of parquet file
9,data\processed\realized_variance_panel_v0_1.csv,1.835705,CSV duplicate of parquet file
10,data\processed\trade_candidate_pricing_v0_1.csv,1.422599,CSV duplicate of parquet file
11,data\processed\trade_candidate_panel_v0_1.csv,1.031248,CSV duplicate of parquet file
12,data\processed\richest_vrp_tenor_by_date_v0_1.csv,1.012476,CSV duplicate of parquet file


Safety checks passed.


In [7]:
# ============================================================
# Execute deletion and save log
# ============================================================

delete_log_rows = []

for _, row in delete_df.iterrows():
    rel_path = str(row["relative_path"]).replace("/", "\\")
    file_path = PROJECT_ROOT / rel_path
    
    log_row = {
        "relative_path": rel_path,
        "file_path": str(file_path),
        "size_mb_reported": row["size_mb"],
        "cleanup_reason": row["cleanup_reason"],
        "delete_attempted": True,
        "delete_success": False,
        "error": "",
    }
    
    try:
        if file_path.exists() and file_path.is_file():
            actual_size_mb = file_path.stat().st_size / (1024 * 1024)
            log_row["size_mb_actual"] = actual_size_mb
            
            file_path.unlink()
            log_row["delete_success"] = True
        else:
            log_row["delete_success"] = False
            log_row["error"] = "file_not_found_or_not_file"
            log_row["size_mb_actual"] = np.nan
    
    except Exception as e:
        log_row["delete_success"] = False
        log_row["error"] = str(e)
        log_row["size_mb_actual"] = np.nan
    
    delete_log_rows.append(log_row)

delete_log_df = pd.DataFrame(delete_log_rows)

delete_log_df.to_csv(DELETE_LOG_CSV, index=False)

print("Deletion log saved:", DELETE_LOG_CSV)

print("\nDelete success counts:")
display(delete_log_df["delete_success"].value_counts().rename("count").to_frame())

print("\nDeleted size MB:")
display(
    delete_log_df
    .groupby("delete_success")["size_mb_actual"]
    .sum()
    .rename("size_mb")
    .to_frame()
)

display(delete_log_df)

Deletion log saved: C:\Users\patri\vrp_project\data\audit\project_cleanup_delete_log_v0_1.csv

Delete success counts:


,count
delete_success,
True,15



Deleted size MB:


,size_mb
delete_success,
True,68.323206


,relative_path,file_path,size_mb_reported,cleanup_reason,delete_attempted,delete_success,error,size_mb_actual
0,data\processed\trade_signal_panel_v0_1.csv,C:\Users\patri\vrp_project\data\processed\trad...,15.642781,CSV duplicate of parquet file,True,True,,15.642781
1,data\processed\vrp_zscore_panel_v0_1.csv,C:\Users\patri\vrp_project\data\processed\vrp_...,14.564443,CSV duplicate of parquet file,True,True,,14.564443
2,data\processed\vrp_panel_v0_1.csv,C:\Users\patri\vrp_project\data\processed\vrp_...,12.067234,CSV duplicate of parquet file,True,True,,12.067234
3,data\processed\vix_term_structure_history_v0_7...,C:\Users\patri\vrp_project\data\processed\vix_...,6.958735,CSV duplicate of parquet file,True,True,,6.958735
4,data\processed\archive\vix_term_structure_hist...,C:\Users\patri\vrp_project\data\processed\arch...,6.196209,CSV duplicate of parquet file,True,True,,6.196209
5,data\processed\vix_term_structure_history.csv,C:\Users\patri\vrp_project\data\processed\vix_...,5.400764,CSV duplicate of parquet file,True,True,,5.400764
6,data\processed\realized_variance_panel_v0_1.csv,C:\Users\patri\vrp_project\data\processed\real...,1.835705,CSV duplicate of parquet file,True,True,,1.835705
7,data\processed\trade_candidate_pricing_v0_1.csv,C:\Users\patri\vrp_project\data\processed\trad...,1.422599,CSV duplicate of parquet file,True,True,,1.422599
8,data\processed\trade_candidate_panel_v0_1.csv,C:\Users\patri\vrp_project\data\processed\trad...,1.031248,CSV duplicate of parquet file,True,True,,1.031248
9,data\processed\richest_vrp_tenor_by_date_v0_1.csv,C:\Users\patri\vrp_project\data\processed\rich...,1.012476,CSV duplicate of parquet file,True,True,,1.012476
